# Testing trained RL agents

In [1]:
import tensorflow as tf
import configparser
import numpy as np
import os

# Load the RL model from the directory
directory = "drl_results/n7_noisy_stepn7_off_size120"
# Find the config file in the directory

for file in os.listdir(directory):
    if file.endswith(".ini"):
        config_file = file
        break
    
config = configparser.ConfigParser()
# Read the config file
config.read(directory + "/" + config_file)

# Copy the config file to the current directory
current_directory = os.getcwd()
os.system(f"cp {directory}/{config_file} {current_directory}")


checkpoint_prefix = directory+'/model/model.ckpt'

# Load the model from the checkpoint
with tf.compat.v1.Session() as sess:
    # Restore the graph structure from the .meta file
    saver = tf.compat.v1.train.import_meta_graph(checkpoint_prefix + ".meta")
    
    # Restore the weights from the checkpoint
    saver.restore(sess, checkpoint_prefix)
    
    
    # The model is now loaded into the session
    print("Model restored successfully!")

2025-05-13 15:29:22.429983: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-13 15:29:22.433814: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-13 15:29:22.482532: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-13 15:29:23.239423: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


INFO:tensorflow:Restoring parameters from drl_results/n7_noisy_stepn7_off_size120/model/model.ckpt
Model restored successfully!


2025-05-13 15:29:28.533538: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:282] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-05-13 15:29:28.564390: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled


In [2]:
from state_env import State  # module with environment and dynamics

from RL_brain_pi_deep import DQNPrioritizedReplay  # sumtree version of DQN

Instructions for updating:
non-resource variables are not supported in the long term
Seed used: 1747160968


In [3]:

env = State()  # environment

[[[  0.   1.   0.   0.   0.   0.   0.]
  [  1.   0.   1.   0.   0.   0.   0.]
  [  0.   1.   0.   1.   0.   0.   0.]
  [  0.   0.   1.   0.   1.   0.   0.]
  [  0.   0.   0.   1.   0.   1.   0.]
  [  0.   0.   0.   0.   1.   0.   1.]
  [  0.   0.   0.   0.   0.   1.   0.]]

 [[100.   1.   0.   0.   0.   0.   0.]
  [  1.   0.   1.   0.   0.   0.   0.]
  [  0.   1.   0.   1.   0.   0.   0.]
  [  0.   0.   1.   0.   1.   0.   0.]
  [  0.   0.   0.   1.   0.   1.   0.]
  [  0.   0.   0.   0.   1.   0.   1.]
  [  0.   0.   0.   0.   0.   1.   0.]]

 [[  0.   1.   0.   0.   0.   0.   0.]
  [  1. 100.   1.   0.   0.   0.   0.]
  [  0.   1.   0.   1.   0.   0.   0.]
  [  0.   0.   1.   0.   1.   0.   0.]
  [  0.   0.   0.   1.   0.   1.   0.]
  [  0.   0.   0.   0.   1.   0.   1.]
  [  0.   0.   0.   0.   0.   1.   0.]]

 [[  0.   1.   0.   0.   0.   0.   0.]
  [  1.   0.   1.   0.   0.   0.   0.]
  [  0.   1. 100.   1.   0.   0.   0.]
  [  0.   0.   1.   0.   1.   0.   0.]
  [  0.   0.   0.  

In [4]:

# Inspect the graph to find input and output tensor names
for op in tf.compat.v1.get_default_graph().get_operations():
    print(op.name)

s
Q_target
eval_net/l1/w1/Initializer/random_normal/shape
eval_net/l1/w1/Initializer/random_normal/mean
eval_net/l1/w1/Initializer/random_normal/stddev
eval_net/l1/w1/Initializer/random_normal/RandomStandardNormal
eval_net/l1/w1/Initializer/random_normal/mul
eval_net/l1/w1/Initializer/random_normal
eval_net/l1/w1
eval_net/l1/w1/Assign
eval_net/l1/w1/read
eval_net/l1/b1/Initializer/Const
eval_net/l1/b1
eval_net/l1/b1/Assign
eval_net/l1/b1/read
eval_net/l1/MatMul
eval_net/l1/add
eval_net/l1/Relu
eval_net/l15/w15/Initializer/random_normal/shape
eval_net/l15/w15/Initializer/random_normal/mean
eval_net/l15/w15/Initializer/random_normal/stddev
eval_net/l15/w15/Initializer/random_normal/RandomStandardNormal
eval_net/l15/w15/Initializer/random_normal/mul
eval_net/l15/w15/Initializer/random_normal
eval_net/l15/w15
eval_net/l15/w15/Assign
eval_net/l15/w15/read
eval_net/l15/b15/Initializer/Const
eval_net/l15/b15
eval_net/l15/b15/Assign
eval_net/l15/b15/read
eval_net/l15/MatMul
eval_net/l15/add
ev

In [5]:
input_tensor = tf.compat.v1.get_default_graph().get_tensor_by_name("s:0")
output_tensor = tf.compat.v1.get_default_graph().get_tensor_by_name("eval_net/l2/add:0")
n = 7

In [6]:
with tf.compat.v1.Session() as sess:
    # Restore the graph structure and weights again
    saver.restore(sess, checkpoint_prefix)

    number_of_episodes = 1000
    lth = config.getint(
            "system_parameters", "max_t_steps"
        )
    
    print("max_t_steps: ", lth)

    best_action_sequences = [[] for i in range(0, 10)]
    best_fidelities = np.zeros(10)

    actionspace = []  # store successful actions
    Qvalue = []  # total reward
    fid_max_vector = []  # max. fidelity in each episode
    t_fid_max_vector = []  # time of max. fidelity
    fid_end_vector = []  # final fidelity
    t_end_vector = []  # time of final fidelity
    success_action_sequences = []  # store successful success_action_seq

    for episode in range(number_of_episodes):
        # Generate a complex normalized vector of 16 components
        observation = env.random_reset()

        print(observation)

        
        
        newaction = []
        Q = 0
        fid_max = 0
        t_fid_max = 0

        for i in range(lth):  # episode maximum length
            # Use the loaded model to predict the action
            # Correct the shape of the observation before feeding it to the model
            predicted_action = sess.run(output_tensor, feed_dict={input_tensor: np.expand_dims(observation, axis=0)})
            predicted_action = np.argmax(predicted_action)
            newaction.append(predicted_action)

            observation_, reward, done, fidelity = env.step(predicted_action)

            Q += reward  # total reward

            # Save max. fidelity value
            if fidelity > fid_max:
                fid_max = fidelity
                t_fid_max = i

            if done:  # fidelity(reward) larger than threshold
                newaction += [0 for xx in range(lth - len(newaction))]
                print(str(i + 1) + "  " + str(episode) + "  " + str(fidelity))
                actionspace.append(newaction)
                Qvalue.append(Q)
                fid_max_vector.append(fid_max)
                fid_end_vector.append(fidelity)
                t_fid_max_vector.append(t_fid_max)
                t_end_vector.append(i + 1)

                if fid_max > 0.9:
                    success_action_sequences.append(newaction)

                    break

            observation = observation_  # Update current state

        if i == lth - 1:
            actionspace.append(newaction)
            fid_max_vector.append(fid_max)
            fid_end_vector.append(fidelity)
            t_fid_max_vector.append(t_fid_max)
            t_end_vector.append(i + 1)
            Qvalue.append(Q)

            if fid_max > 0.9:
                success_action_sequences.append(newaction)

        if fid_max > min(best_fidelities):
            idx = np.argmin(best_fidelities)
            best_fidelities[idx] = fid_max
            best_action_sequences[idx] = newaction

        print(
            "Episode: %d, Step: %d, Max. Fidelity: %.4f"
            % (episode, i, fid_max)
        )
        episodes = np.arange(0, number_of_episodes)

    print("Mean of fid_max_vector:", np.mean(fid_max_vector))

INFO:tensorflow:Restoring parameters from drl_results/n7_noisy_stepn7_off_size120/model/model.ckpt
max_t_steps:  35
[ 0.99968336  0.          0.00173392  0.          0.02127475  0.
  0.00790072  0.          0.00651996  0.         -0.0078059   0.
  0.00341945  0.        ]
30  0  0.9640023371381716
Episode: 0, Step: 29, Max. Fidelity: 0.9640
[ 9.99760365e-01  0.00000000e+00  6.18185193e-03  0.00000000e+00
  4.94266851e-04  0.00000000e+00 -2.15920142e-03  0.00000000e+00
  1.09294154e-02  0.00000000e+00 -1.74249675e-02  0.00000000e+00
 -3.60682414e-03  0.00000000e+00]
30  1  0.9687102307278953
Episode: 1, Step: 29, Max. Fidelity: 0.9687
[ 0.99969685  0.         -0.01268971  0.          0.01671298  0.
 -0.00756191  0.         -0.002086    0.          0.00341669  0.
 -0.0096259   0.        ]
30  2  0.9646330308174562
Episode: 2, Step: 29, Max. Fidelity: 0.9646
[ 9.99630429e-01  0.00000000e+00 -1.87564933e-02  0.00000000e+00
 -6.45278049e-03  0.00000000e+00 -9.58234922e-04  0.00000000e+00
 -7